In [3]:
import pandas as pd
import numpy as np  
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

dp=pd.read_csv('train/diabetes.csv')


# Split first, before any scaling
X = dp.drop('diabetes', axis=1)
y = dp['diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

#  Only scale the numeric columns that need it
cols_to_scale = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']

scaler = StandardScaler()

# fit_transform on train, transform (not fit) on test
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# applying OneHotEncoder to categorical columns
categorical_cols=['gender','smoking_history']
encoder = OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore')

# Fit on train only
encoder.fit(X_train[categorical_cols])

# Transform both
X_train_encoded = pd.DataFrame(
    encoder.transform(X_train[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    encoder.transform(X_test[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_test.index
)

# Drop original categorical columns and join the encoded ones
X_train = X_train.drop(columns=categorical_cols).join(X_train_encoded)
X_test = X_test.drop(columns=categorical_cols).join(X_test_encoded)


lr = LogisticRegression(class_weight={0:1,1:3})
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)


print("Accuracy", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))



Accuracy 0.9482
[[17688   612]
 [  424  1276]]
              precision    recall  f1-score   support

           0       0.98      0.97      0.97     18300
           1       0.68      0.75      0.71      1700

    accuracy                           0.95     20000
   macro avg       0.83      0.86      0.84     20000
weighted avg       0.95      0.95      0.95     20000



In [4]:
X_train.sample(50)

,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,gender_Male,gender_Other,smoking_history_current,smoking_history_ever,smoking_history_former,smoking_history_never,smoking_history_not current
11883,-1.549089,0,0,-1.804996,-1.893645,-0.297058,1.0,0.0,0.0,0.0,0.0,0.0,0.0
90507,0.448269,0,0,0.419088,0.628060,0.170207,0.0,0.0,0.0,0.0,0.0,1.0,0.0
79182,-1.194003,0,0,-0.540114,-1.893645,-0.198686,0.0,0.0,0.0,0.0,0.0,0.0,0.0
65454,0.270726,0,0,-0.681661,0.628060,1.522815,0.0,0.0,0.0,0.0,0.0,1.0,0.0
12427,0.803355,0,0,0.819634,0.628060,0.416136,1.0,0.0,0.0,0.0,1.0,0.0,0.0
5537,1.202827,0,1,0.318199,2.495989,2.506531,1.0,0.0,1.0,0.0,0.0,0.0,0.0
34137,-0.439445,0,0,0.336268,0.908249,1.522815,1.0,0.0,0.0,0.0,0.0,0.0,0.0
58637,-0.750146,0,0,0.000472,0.908249,-1.428330,0.0,0.0,0.0,0.0,0.0,1.0,0.0
13614,-1.637860,0,0,0.000472,0.908249,0.514507,1.0,0.0,0.0,0.0,0.0,0.0,0.0
21188,-1.016460,0,0,-0.844288,-0.679491,0.539100,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [5]:
import joblib, json

joblib.dump(lr, "model/model.pkl")
joblib.dump(scaler, "model/scaler.pkl")
joblib.dump(encoder, "model/encoder.pkl")
json.dump(list(X_train.columns), open("model/feature_order.json", "w"))